In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"
system_prompt = """
あなたはとても簡潔にソリューションを提示できる優秀なエンジニアです。
"""

In [ ]:
# tool
from anthropic.types import ToolParam
from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

def add_duration_to_datetime(datetime_str, days=0, hours=0, minutes=0, seconds=0, date_format="%Y-%m-%d %H:%M:%S"):
    dt = datetime.strptime(datetime_str, date_format)
    result = dt + timedelta(days=days, hours=hours, minutes=minutes, seconds=seconds)
    return result.strftime(date_format)

# 単にjsonオブジェクトを定義してもよいが、ToolParamでラップするとコードの堅牢性が上がるらしい(?)
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "現在の日時を指定したフォーマットの文字列で取得します。",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "datetime.strftime に渡す日時フォーマット文字列",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": "指定した日時に期間(日数・時間・分・秒)を加算(または減算)した結果を文字列で取得します。",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "基準となる日時の文字列(date_formatに従う)"
            },
            "days": {
                "type": "integer",
                "description": "加算する日数(負の値で減算)",
                "default": 0
            },
            "hours": {
                "type": "integer",
                "description": "加算する時間数(負の値で減算)",
                "default": 0
            },
            "minutes": {
                "type": "integer",
                "description": "加算する分数(負の値で減算)",
                "default": 0
            },
            "seconds": {
                "type": "integer",
                "description": "加算する秒数(負の値で減算)",
                "default": 0
            },
            "date_format": {
                "type": "string",
                "description": "datetime のパース・フォーマットに使う日時フォーマット文字列",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": ["datetime_str"]
    }
})

def run_tool(tool_name, tool_input):
    # ツール名 -> 実行する関数 のマッピング
    tool_functions = {
        "get_current_datetime": get_current_datetime,
        "add_duration_to_datetime": add_duration_to_datetime,
    }
    return tool_functions[tool_name](**tool_input)
    

In [ ]:
# web_search はサーバーサイドツールなので input_schema は不要で、run_tool にも登録しない
# (Anthropic側で検索が実行され、結果は tool_use ではなく web_search_tool_result ブロックとしてレスポンスに含まれる)
web_search_tool_schema = ToolParam({
    "type": "web_search_20260209",
    "name": "web_search",
    "max_uses": 5,  # 1リクエストあたりの検索回数上限(任意)
})

In [ ]:
import json

# Claude メッセージヘルパー
def to_user_message(content):
    user_message = {"role": "user", "content": content}
    return user_message

def to_assistant_message(content):
    assistant_message = {"role": "assistant", "content": content}
    return assistant_message

def execute_tool_use_blocks(content_blocks):
    """response.content内のtool_useブロックを実行し、tool_resultブロックのリストを返す"""
    tool_results = []
    for block in content_blocks:
        if block.type == "tool_use":
            try :
                result = run_tool(block.name, block.input)
                isError = False
            except Exception as e:
                result = f"Error: {e}"
                isError = True
            
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result),
                "is_error": isError
            })
    return tool_results

def chat(messages, effort="medium"):
    return client.messages.stream(
        model=model,
        max_tokens=8192,
        # temperature, top_p, top_k は adaptive thinking では指定不可(400になる)
        messages=messages,
        system=system_prompt,
        thinking={
            "type": "adaptive",  # claude-sonnet-5 では budget_tokens は指定不可。深さは output_config.effort で制御する
            "display": "summarized"  # デフォルトは "omitted" で thinking テキストが空になるため明示指定
        },
        output_config={"effort": effort},  # low | medium | high | xhigh | max
        tools=[get_current_datetime_schema, add_duration_to_datetime_schema, web_search_tool_schema]
    )

def stream_and_print(stream):
    """ストリームを消費しつつ、thinking(思考過程)とtext(回答)を区別して標準出力に表示する"""
    current_block_type = None
    previous_block_type = None
    for event in stream:
        if event.type == "content_block_start":
            current_block_type = event.content_block.type
            if current_block_type != previous_block_type:
                previous_block_type = current_block_type
                print()
                print()
                if current_block_type == "thinking":
                    print("Thinking...")
                elif current_block_type == "text":
                    print("Expressing...")
        elif event.type == "content_block_delta":
            if event.delta.type == "thinking_delta":
                print(event.delta.thinking, end="", flush=True)
            elif event.delta.type == "text_delta":
                print(event.delta.text, end="", flush=True)
    return stream.get_final_message()

In [ ]:
import base64

def build_pdf_document_block(pdf_path, title=None, enable_citations=True):
    """PDFファイルを読み込み、base64エンコードしたdocumentコンテンツブロックを作成する"""
    with open(pdf_path, "rb") as f:
        file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

    return {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
        "title": title or pdf_path,
        "citations": {"enabled": enable_citations},
    }

In [ ]:
# 8_earth.pdf をClaudeに渡して質問する例
messages = []

pdf_block = build_pdf_document_block("8_earth.pdf", title="Earth Article")
user_message = to_user_message([
    pdf_block,
    {"type": "text", "text": "地球の大気と海洋はどのように形成されたか、日本語で答えてください。"},
])
messages.append(user_message)

with chat(messages) as stream:
    response = stream_and_print(stream)
print()
assistant_message = to_assistant_message(response.content)
messages.append(assistant_message)

# レスポンス中の引用(citations)部分だけを表示する
print()
print('--------- citations')
for block in response.content:
    if block.type == "text" and block.citations:
        for citation in block.citations:
            print(citation)